# True bHUB — Bayesian Inference of Hub Nodes Across Multiple Networks
## AMD Co-expression Networks: MGS1 (normal/early) vs MGS4 (advanced AMD)

**Reference:** Kim J, Do K-A, Ha MJ, Peterson CB. (2019).  
*Bayesian Inference of Hub Nodes Across Multiple Networks.*  
Biometrics 75(1): 172–182. DOI: 10.1111/biom.12958

---
### Model
For $K=2$ groups (MGS1, MGS4), $p=81$ genes, $n_k$ samples:

$$\mathbf{x}_{k,i} \sim \mathcal{N}(0,\,\Omega_k^{-1}) \quad \text{(likelihood)}$$

**Precision prior (SSSL, Wang 2015):**
$$\omega_{k,ij} \mid g_{k,ij} \sim \mathcal{N}(0,\, v_{k,ij}^2), \quad v_{k,ij} = \begin{cases} v_1 & g_{k,ij}=1 \\ v_0 & g_{k,ij}=0 \end{cases}$$

**Edge prior** (depends on hub status):
$$\pi_{k,ij} = \beta_1(1-h_{k,i})(1-h_{k,j}) + \beta_2\{1-(1-h_{k,i})(1-h_{k,j})\}$$

**Hub prior** (links groups via Ising-type prior):
$$P(h_{k,i}\mid \tau,\eta,\mathbf{h}_{-k,i}) \propto \exp\!\left(h_{k,i}\left(\eta + 2\tau\sum_{m\neq k}h_{m,i}\right)\right)$$

**Key output:** Posterior probability of hub $\text{PPI}_{k,i} = P(h_{k,i}=1\mid\mathbf{X})$  
Gene declared hub if $\text{PPI}_{k,i} \geq 0.5$ (posterior median graph criterion).

### MCMC algorithm
1. Column-wise block Gibbs for $\Omega_k, G_k$ (Wang 2015)
2. Closed-form Bernoulli for $h_{k,i}$
3. Conjugate Beta for $\beta_2$
4. Log-normal Metropolis for $\tau$

**Runtime:** ~2–4 min on Colab (Linux + MKL-linked numpy).

In [ ]:
# ── Install (Colab already has most of these) ─────────────────────────────────
!pip install numpy scipy matplotlib seaborn networkx pandas tqdm -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import networkx as nx
from tqdm.notebook import tqdm
import warnings, io
warnings.filterwarnings('ignore')

np.random.seed(2024)
print('Libraries loaded.')

## 1 · Load Data

In [ ]:
from google.colab import files

print('Upload aak100_cpmdat.csv ...')
uploaded = files.upload()                          # browse and select the file
key = list(uploaded.keys())[0]
df  = pd.read_csv(io.BytesIO(uploaded[key]), index_col=0)

feature_cols = [c for c in df.columns if c != 'mgs_level']
gene_names   = np.array(feature_cols)
p            = len(gene_names)

from sklearn.preprocessing import StandardScaler
X_mgs1 = df.loc[df['mgs_level'] == 'MGS1', feature_cols].values
X_mgs4 = df.loc[df['mgs_level'] == 'MGS4', feature_cols].values
X1 = StandardScaler().fit_transform(X_mgs1)
X4 = StandardScaler().fit_transform(X_mgs4)

print(f'Genes (p)    : {p}')
print(f'MGS1 samples : {X1.shape[0]}')
print(f'MGS4 samples : {X4.shape[0]}')

## 2 · Hyperparameters
All defaults follow Kim et al. (2019) Section 3.5 and the illustrative example (Section 5).

In [ ]:
# ── Precision matrix prior (Wang 2015) ───────────────────────────────────────
v0_sq  = 0.03          # spike variance (absent edges)
v1_sq  = 3.0           # slab variance (present edges;  100 × v0_sq)
lam    = 1.0           # Exp(lam/2) prior on diagonal elements

# ── Edge inclusion priors ─────────────────────────────────────────────────────
beta1  = 0.001         # prior prob for non-hub edge
a_b2   = 1.0           # Beta(a,b) prior on beta2
b_b2   = 39.0          # => prior mean beta2 = 1/40 = 0.025  (25x beta1)

# ── Hub indicator prior ───────────────────────────────────────────────────────
# logit^{-1}(eta) = 0.01  =>  eta = log(0.01/0.99)  (1% baseline hub prob)
eta    = float(np.log(0.01 / 0.99))
a_tau  = 1.5           # Gamma(a_tau, b_tau) prior on tau
b_tau  = 0.2

# ── MCMC settings ────────────────────────────────────────────────────────────
N_BURNIN = 1000        # discard these samples
N_ITER   = 2000        # posterior inference samples
# For faster testing set N_BURNIN=200, N_ITER=500

# ── Derived constants (precomputed for hot path) ──────────────────────────────
v0, v1             = np.sqrt(v0_sq), np.sqrt(v1_sq)
LOG_V0_OVER_V1     = np.log(v0 / v1)                  # negative
INV_V0_SQ_M_V1_SQ  = 1.0/v0_sq - 1.0/v1_sq           # large positive

K       = 2
data    = [X1, X4]
n_list  = [X1.shape[0], X4.shape[0]]
S_list  = [X.T @ X for X in data]     # sufficient statistics (fixed)

# Precompute index arrays (avoids repeated np.concatenate in hot path)
COL_IDX = [np.concatenate([np.arange(j), np.arange(j+1, p)]) for j in range(p)]

print('Hyperparameters set.')
print(f'  v0={v0:.4f}  v1={v1:.4f}  lam={lam}')
print(f'  beta1={beta1}  E[beta2]={a_b2/(a_b2+b_b2):.4f}')
print(f'  eta={eta:.4f}  (base hub prob={1/(1+np.exp(-eta)):.3f})')
print(f'  MCMC: {N_BURNIN} burn-in + {N_ITER} inference')

## 3 · MCMC Implementation

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Step 1 — Precision matrix + graph update  (Wang 2015 SSSL column-wise Gibbs)
# ═══════════════════════════════════════════════════════════════════════════════
def sweep_precision_graph(k):
    """
    Column-wise Gibbs update for Omega_k and G_k.

    Key formulas (column j):
      C_11 = (Omega_{-j,-j})^{-1}  recovered from Sigma via block-inverse formula
      A    = s_{jj} C_{11} + diag(1/v^2)     [posterior precision for gamma]
      gamma ~ N(-A^{-1} s_{-j,j},  A^{-1})   [off-diagonal of column j]
      delta_j ~ Gamma(n/2+1,  2/(s_{jj}+lam)) [Schur complement]
      omega_{jj} = delta_j + gamma^T C_{11} gamma

    Sigma is updated in O(p^2) using the partitioned inverse formula,
    avoiding an O(p^3) inversion inside the loop.
    """
    S   = S_list[k]
    Om  = Omega[k]
    Sg  = Sigma[k]    # maintained as Om^{-1}
    Vk  = V[k]
    Gk  = G[k]
    hk  = H[k]
    n   = n_list[k]

    for j in range(p):
        idx  = COL_IDX[j]          # integer indices for all i != j
        s22  = S[j, j]
        s12  = S[idx, j]

        # ── C11 = (Omega_{-j,-j})^{-1} from current Sigma ────────────────────
        sig22 = Sg[j, j]
        sig12 = Sg[idx, j]
        Sg11  = Sg[np.ix_(idx, idx)]
        C11   = Sg11 - np.outer(sig12, sig12) / sig22   # O(p^2)

        # ── Posterior precision A = s22*C11 + diag(1/v^2) ────────────────────
        A = s22 * C11
        A[np.diag_indices(p-1)] += 1.0 / Vk[idx, j]

        # ── Sample gamma ~ N(mu, A^{-1}) via Cholesky ────────────────────────
        try:
            L     = np.linalg.cholesky(A)
            mu    = np.linalg.solve(A, -s12)
            gamma = mu + np.linalg.solve(L.T, np.random.randn(p-1))
        except np.linalg.LinAlgError:
            gamma = Om[idx, j]     # keep current value on numerical failure

        # ── Sample delta_j ~ Gamma(n/2+1, 2/(s22+lam)) ───────────────────────
        delta_j = np.random.gamma(n / 2.0 + 1.0, 2.0 / (s22 + lam))

        # ── Update Omega ──────────────────────────────────────────────────────
        C11g     = C11 @ gamma
        Om[j, j] = delta_j + gamma @ C11g
        Om[idx, j] = gamma
        Om[j, idx] = gamma

        # ── Update Sigma via partitioned inverse (O(p^2), no inversion) ──────
        inv_dj        = 1.0 / delta_j
        Sg[j, j]      = inv_dj
        Sg[idx, j]    = -C11g * inv_dj
        Sg[j, idx]    = -C11g * inv_dj
        Sg[np.ix_(idx, idx)] = C11 + np.outer(C11g, C11g) * inv_dj

        # ── Update graph edges (i, j) — vectorised ───────────────────────────
        pi_col = np.where(hk[idx] | bool(hk[j]), beta2, beta1)
        log_bf = (np.log(pi_col + 1e-15) - np.log(1.0 - pi_col + 1e-15)
                  + LOG_V0_OVER_V1
                  + 0.5 * gamma**2 * INV_V0_SQ_M_V1_SQ)
        p_edge = 1.0 / (1.0 + np.exp(-np.clip(log_bf, -30.0, 30.0)))
        g_col  = (np.random.rand(p-1) < p_edge).astype(np.int8)

        Gk[idx, j] = g_col
        Gk[j, idx] = g_col
        new_v = np.where(g_col, v1_sq, v0_sq)
        Vk[idx, j] = new_v
        Vk[j, idx] = new_v


# ═══════════════════════════════════════════════════════════════════════════════
# Step 2 — Hub indicator update  (closed-form Bernoulli)
# ═══════════════════════════════════════════════════════════════════════════════
def sweep_hubs():
    """
    For each gene i in each group k:

      log-odds = sum_{j: h_{k,j}=0} [ g_{ij} log(beta2/beta1)
                                     + (1-g_{ij}) log((1-beta2)/(1-beta1)) ]
               + eta + 2*tau * h_{m,i}    (m = other group)

      h_{k,i} ~ Bernoulli( sigmoid(log-odds) )

    Only edges to non-hub nodes contribute (edges to hubs carry beta2
    regardless of h_{k,i}, so they cancel in the log-odds).
    """
    lbr  = np.log(beta2  + 1e-15) - np.log(beta1  + 1e-15)   # log(b2/b1)
    lnbr = np.log(1.0 - beta2 + 1e-15) - np.log(1.0 - beta1 + 1e-15)

    for k in range(K):
        Gk      = G[k].astype(float)
        hk      = H[k]
        h_other = H[1 - k]     # K=2: the other group

        for i in range(p):
            # Neighbours that are NOT hubs (only these shift the log-odds)
            nh           = (hk == 0)
            nh[i]        = False
            g_row        = Gk[i, nh]
            delta_like   = g_row.sum() * lbr + (nh.sum() - g_row.sum()) * lnbr
            delta_prior  = eta + 2.0 * tau * float(h_other[i])
            log_odds     = delta_like + delta_prior
            prob         = 1.0 / (1.0 + np.exp(-np.clip(log_odds, -30.0, 30.0)))
            hk[i]        = np.random.binomial(1, prob)


# ═══════════════════════════════════════════════════════════════════════════════
# Step 3 — beta2 update  (conjugate Beta)
# ═══════════════════════════════════════════════════════════════════════════════
def update_beta2():
    """
    beta2 | G, h ~ Beta(a + #hub-edges present, b + #hub-edges absent)
    where a hub-edge is any (i,j) with h_{k,i}=1 or h_{k,j}=1.
    """
    n_pres = n_abs = 0
    for k in range(K):
        Gk   = G[k]
        hk   = H[k]
        hmask = (hk[:, None] | hk[None, :]).astype(bool)
        np.fill_diagonal(hmask, False)
        upper  = np.triu(hmask, k=1)
        n_pres += int(Gk[upper].sum())
        n_abs  += int(upper.sum()) - n_pres
    return float(np.random.beta(a_b2 + n_pres, b_b2 + max(n_abs, 0)))


# ═══════════════════════════════════════════════════════════════════════════════
# Step 4 — tau update  (log-normal Metropolis-within-Gibbs)
# ═══════════════════════════════════════════════════════════════════════════════
def log_ph(tau_val):
    """
    log p(H | tau, eta)  summed over all p genes.
    For K=2: C(eta,tau) = 1 + 2 exp(eta) + exp(2eta + 2tau)
    """
    log_C = np.log(1.0 + 2.0*np.exp(eta) + np.exp(2.0*eta + 2.0*tau_val))
    h0, h1 = H[0].astype(float), H[1].astype(float)
    # For each gene i: eta*(h0+h1) + 2*tau*h0*h1 - log_C
    return float(np.sum(eta*(h0+h1) + 2.0*tau_val*(h0*h1) - log_C))


def update_tau(tau_current, sigma_prop=0.3):
    """
    Log-normal random-walk Metropolis for tau (constrained to tau > 0).
    Jacobian of the log-transform gives an extra log(tau_p/tau_c) term.
    """
    lt_c  = np.log(tau_current)
    lt_p  = lt_c + sigma_prop * np.random.randn()
    tau_p = np.exp(lt_p)
    # log MH ratio  (target × Jacobian; proposal is symmetric on log scale)
    log_alpha = (
        log_ph(tau_p)   - log_ph(tau_current)
        + a_tau*(lt_p   - lt_c)
        - b_tau*(tau_p  - tau_current)
    )
    if np.log(np.random.uniform()) < log_alpha:
        return tau_p, True
    return tau_current, False


print('MCMC functions defined.')

## 4 · Initialise Chain

In [ ]:
# ── State variables ───────────────────────────────────────────────────────────
Omega = [np.eye(p)                     for _ in range(K)]   # precision matrices
Sigma = [np.eye(p)                     for _ in range(K)]   # covariance = Omega^{-1}
G     = [np.zeros((p,p), dtype=np.int8) for _ in range(K)]  # adjacency matrices
V     = [np.full((p,p), v0_sq)         for _ in range(K)]   # edge variances
H     = [np.zeros(p,   dtype=np.int8)  for _ in range(K)]   # hub indicators

beta2 = float(a_b2 / (a_b2 + b_b2))   # prior mean
tau   = float(a_tau / b_tau)           # prior mean

# Ensure V diagonal is 0 (no self-edges)
for k in range(K):
    np.fill_diagonal(V[k], 0.0)

# ── Storage arrays (post-burn-in only) ────────────────────────────────────────
ppi_samples   = np.zeros((K, p, N_ITER), dtype=np.int8)   # hub indicators
eppi_sum      = [np.zeros((p,p)) for _ in range(K)]       # edge PPI running sum
beta2_trace   = np.zeros(N_BURNIN + N_ITER)
tau_trace     = np.zeros(N_BURNIN + N_ITER)

print('Chain initialised.')
print(f'Storage: hub samples {ppi_samples.nbytes/1e6:.1f} MB')

## 5 · Run MCMC

In [ ]:
tau_accept = 0
TOTAL      = N_BURNIN + N_ITER

for it in tqdm(range(TOTAL), desc='MCMC'):

    # ── Step 1: precision matrix + graph ─────────────────────────────────────
    for k in range(K):
        sweep_precision_graph(k)

    # ── Step 2: hub indicators ────────────────────────────────────────────────
    sweep_hubs()

    # ── Step 3: beta2 ─────────────────────────────────────────────────────────
    beta2 = update_beta2()

    # ── Step 4: tau ───────────────────────────────────────────────────────────
    tau, acc  = update_tau(tau)
    tau_accept += acc

    beta2_trace[it] = beta2
    tau_trace[it]   = tau

    # ── Post burn-in: store samples ───────────────────────────────────────────
    if it >= N_BURNIN:
        t = it - N_BURNIN
        for k in range(K):
            ppi_samples[k, :, t] = H[k]
            eppi_sum[k]         += G[k]

print(f'\nDone.  tau acceptance rate: {tau_accept/TOTAL:.3f}  (target 0.20–0.50)')

## 6 · Convergence Diagnostics

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 7))
fig.suptitle('MCMC Convergence Diagnostics', fontsize=13, fontweight='bold')

burn_line = dict(color='red', linestyle='--', linewidth=1, alpha=0.7, label='Burn-in end')

# tau trace
ax = axes[0, 0]
ax.plot(tau_trace, lw=0.6, color='steelblue')
ax.axvline(N_BURNIN, **burn_line); ax.legend(fontsize=8)
ax.set_xlabel('Iteration'); ax.set_ylabel('tau'); ax.set_title('Trace: tau')
ax.grid(alpha=0.3)

# beta2 trace
ax = axes[0, 1]
ax.plot(beta2_trace, lw=0.6, color='darkorange')
ax.axvline(N_BURNIN, **burn_line); ax.legend(fontsize=8)
ax.set_xlabel('Iteration'); ax.set_ylabel('beta2'); ax.set_title('Trace: beta2')
ax.grid(alpha=0.3)

# Posterior density of tau (post burn-in)
ax = axes[1, 0]
ax.hist(tau_trace[N_BURNIN:], bins=40, color='steelblue', alpha=0.7, density=True)
ax.set_xlabel('tau'); ax.set_ylabel('Density'); ax.set_title('Posterior: tau')
ax.grid(alpha=0.3)

# Posterior density of beta2 (post burn-in)
ax = axes[1, 1]
ax.hist(beta2_trace[N_BURNIN:], bins=40, color='darkorange', alpha=0.7, density=True)
ax.set_xlabel('beta2'); ax.set_ylabel('Density'); ax.set_title('Posterior: beta2')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('bhub_diagnostics.png', dpi=130, bbox_inches='tight')
plt.show()

## 7 · Posterior Inference

In [ ]:
# ── Posterior probability of hub (PPI) ────────────────────────────────────────
# PPI_{k,i} = P(h_{k,i}=1 | X)  approximated by the MCMC mean
PPI      = ppi_samples.mean(axis=2)     # shape (K, p)
PPI_MGS1 = PPI[0];  PPI_MGS4 = PPI[1]

# ── Hub classification  (posterior median: PPI >= 0.5) ────────────────────────
HUB_THRESH = 0.5
hub1     = gene_names[PPI_MGS1 >= HUB_THRESH]
hub4     = gene_names[PPI_MGS4 >= HUB_THRESH]
cohubs   = np.intersect1d(hub1, hub4)
only_mgs1 = np.setdiff1d(hub1, hub4)
only_mgs4 = np.setdiff1d(hub4, hub1)

print('=' * 55)
print(f'Hub threshold        : PPI >= {HUB_THRESH}')
print(f'Hub genes – MGS1     : {len(hub1)}')
print(f'Hub genes – MGS4     : {len(hub4)}')
print(f'Co-hub genes (shared): {len(cohubs)}')
print(f'MGS1-specific        : {len(only_mgs1)}')
print(f'MGS4-specific        : {len(only_mgs4)}')
print('=' * 55)

if len(cohubs):
    print('Co-hub genes:', ', '.join(cohubs))
if len(only_mgs4):
    print('MGS4-specific hub genes:', ', '.join(only_mgs4))
if len(only_mgs1):
    print('MGS1-specific hub genes:', ', '.join(only_mgs1))

# ── Posterior median graphs ────────────────────────────────────────────────────
# Edge PPI = fraction of post-burn-in iterations the edge was included
ePPI     = [eppi_sum[k] / N_ITER for k in range(K)]
G_median = [(ePPI[k] >= 0.5).astype(int) for k in range(K)]
for k in range(K):
    np.fill_diagonal(G_median[k], 0)

degree1 = G_median[0].sum(axis=1)
degree4 = G_median[1].sum(axis=1)
print(f'\nMedian-graph edges – MGS1: {G_median[0].sum()//2}')
print(f'Median-graph edges – MGS4: {G_median[1].sum()//2}')

# ── Results table ─────────────────────────────────────────────────────────────
def hub_status(g, c):
    return 'Co-hub' if g and c else ('MGS1-only' if g else ('MGS4-only' if c else 'Non-hub'))

results = pd.DataFrame({
    'Gene'       : gene_names,
    'PPI_MGS1'   : np.round(PPI_MGS1, 4),
    'PPI_MGS4'   : np.round(PPI_MGS4, 4),
    'Hub_MGS1'   : (PPI_MGS1 >= HUB_THRESH).astype(int),
    'Hub_MGS4'   : (PPI_MGS4 >= HUB_THRESH).astype(int),
    'Degree_MGS1': degree1,
    'Degree_MGS4': degree4,
})
results['Status'] = results.apply(
    lambda r: hub_status(r.Hub_MGS1, r.Hub_MGS4), axis=1)
results = results.sort_values('PPI_MGS4', ascending=False).reset_index(drop=True)

print('\nTop 20 genes by MGS4 PPI:')
print(results.head(20).to_string(index=False))

## 8 · Visualisation

In [ ]:
COLOR = {'Co-hub':'#d62728', 'MGS1-only':'#1f77b4',
         'MGS4-only':'#ff7f0e', 'Non-hub':'#cccccc'}

fig = plt.figure(figsize=(16, 16))
gs  = gridspec.GridSpec(3, 2, figure=fig, hspace=0.50, wspace=0.35)
fig.suptitle(
    f'True bHUB — AMD Co-expression Networks\n'
    f'({N_ITER} post-burn-in samples, PPI threshold {HUB_THRESH})',
    fontsize=13, fontweight='bold')

# ── Panel A: PPI scatter ──────────────────────────────────────────────────────
ax = fig.add_subplot(gs[0, :])
for status in ['Non-hub', 'MGS1-only', 'MGS4-only', 'Co-hub']:
    sub = results[results['Status'] == status]
    ax.scatter(sub['PPI_MGS1'], sub['PPI_MGS4'],
               color=COLOR[status], label=status, s=60,
               alpha=0.80, edgecolors='none', zorder=3)
ax.axvline(HUB_THRESH, color='gray', ls='--', lw=1.2, alpha=0.7)
ax.axhline(HUB_THRESH, color='gray', ls='--', lw=1.2, alpha=0.7)
for _, row in results[results['Status'] != 'Non-hub'].head(12).iterrows():
    ax.annotate(row['Gene'], (row['PPI_MGS1'], row['PPI_MGS4']),
                fontsize=7, xytext=(4,3), textcoords='offset points')
ax.set_xlabel('PPI – MGS1  (posterior probability of being a hub)')
ax.set_ylabel('PPI – MGS4')
ax.set_title('(A)  bHUB Posterior Hub Probabilities\n'
             'Dashed lines = 0.5 threshold  |  '
             'Red=Co-hub · Blue=MGS1-only · Orange=MGS4-only')
ax.legend(loc='upper left', fontsize=9); ax.grid(alpha=0.3)

# ── Panel B: top hub genes bar chart ─────────────────────────────────────────
ax = fig.add_subplot(gs[1, 0])
top = results[results['Status'] != 'Non-hub'].head(16)
if len(top) == 0:
    top = results.head(16)
x = np.arange(len(top)); w = 0.38
ax.bar(x-w/2, top['PPI_MGS1'], w, label='MGS1', color='#1f77b4', alpha=0.85)
ax.bar(x+w/2, top['PPI_MGS4'], w, label='MGS4', color='#ff7f0e', alpha=0.85)
ax.axhline(HUB_THRESH, color='red', ls='--', lw=1.2, label=f'PPI={HUB_THRESH}')
ax.set_xticks(x)
ax.set_xticklabels(top['Gene'], rotation=45, ha='right', fontsize=7)
ax.set_ylabel('PPI'); ax.set_title('(B)  Hub Gene PPIs: MGS1 vs MGS4')
ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.3)

# ── Panel C: hub status pie ───────────────────────────────────────────────────
ax = fig.add_subplot(gs[1, 1])
sc = results['Status'].value_counts()
present = [s for s in ['Co-hub','MGS1-only','MGS4-only','Non-hub'] if s in sc]
ax.pie([sc[s] for s in present], labels=present,
       colors=[COLOR[s] for s in present],
       autopct='%1.0f%%', startangle=90, textprops={'fontsize':9})
ax.set_title('(C)  Gene Hub Status Distribution')

# ── Helper: draw hub sub-network ──────────────────────────────────────────────
def draw_subnet(ax, hub_genes, G_med, k_idx, title, node_color_fn):
    nodes = [g for g in hub_genes if g in gene_names]
    if len(nodes) < 2:
        ax.text(0.5, 0.5, 'No hubs above threshold',
                ha='center', va='center', transform=ax.transAxes)
        ax.set_title(title); ax.axis('off'); return
    idx   = [np.where(gene_names == g)[0][0] for g in nodes]
    sub_A = G_med[np.ix_(idx, idx)]
    Gsub  = nx.from_numpy_array(sub_A)
    nx.relabel_nodes(Gsub, {i: nodes[i] for i in range(len(nodes))}, copy=False)
    pos   = nx.spring_layout(Gsub, seed=42, k=2.0/max(np.sqrt(len(nodes)),1))
    nc    = [node_color_fn(n) for n in Gsub.nodes()]
    ns    = [max(200, Gsub.degree(n)*70) for n in Gsub.nodes()]
    nx.draw_networkx(Gsub, pos=pos, ax=ax, node_size=ns, node_color=nc,
                     edge_color='#aaaaaa', font_size=6, alpha=0.85, width=0.8)
    ax.set_title(title); ax.axis('off')

# ── Panel D: MGS1 hub sub-network ────────────────────────────────────────────
ax = fig.add_subplot(gs[2, 0])
draw_subnet(ax, hub1, G_median[0], 0,
            f'(D)  MGS1 Hub Sub-network  ({len(hub1)} nodes)\nRed=co-hub · Blue=MGS1-only',
            lambda n: '#d62728' if n in cohubs else '#1f77b4')

# ── Panel E: MGS4 hub sub-network ────────────────────────────────────────────
ax = fig.add_subplot(gs[2, 1])
draw_subnet(ax, hub4, G_median[1], 1,
            f'(E)  MGS4 Hub Sub-network  ({len(hub4)} nodes)\nRed=co-hub · Orange=MGS4-only',
            lambda n: '#d62728' if n in cohubs else '#ff7f0e')

plt.savefig('bhub_true_results.png', dpi=130, bbox_inches='tight')
plt.show()
print('Figure saved: bhub_true_results.png')

## 9 · Edge PPI Heatmap (posterior co-expression network)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Edge Posterior Probability of Inclusion (PPI)', fontsize=12, fontweight='bold')

for k, (ax, label) in enumerate(zip(axes, ['MGS1', 'MGS4'])):
    ep = ePPI[k].copy()
    np.fill_diagonal(ep, np.nan)
    sns.heatmap(ep, ax=ax, cmap='YlOrRd', vmin=0, vmax=1,
                xticklabels=False, yticklabels=False, cbar_kws={'label':'Edge PPI'})
    ax.set_title(f'{label}  (median-graph edges: {G_median[k].sum()//2})')

plt.tight_layout()
plt.savefig('bhub_edge_ppi.png', dpi=130, bbox_inches='tight')
plt.show()

## 10 · Save Results

In [ ]:
# ── Per-gene results ──────────────────────────────────────────────────────────
results.to_csv('bhub_true_results.csv', index=False)
print('Saved: bhub_true_results.csv')

# ── Edge PPIs (flattened upper triangle) ──────────────────────────────────────
rows_list = []
for k, label in enumerate(['MGS1','MGS4']):
    ep = ePPI[k]
    for i in range(p):
        for j in range(i+1, p):
            rows_list.append({'Group':label,
                              'Gene_i':gene_names[i], 'Gene_j':gene_names[j],
                              'EdgePPI':round(float(ep[i,j]),4),
                              'InMedianGraph':int(G_median[k][i,j])})
pd.DataFrame(rows_list).to_csv('bhub_true_edges.csv', index=False)
print('Saved: bhub_true_edges.csv')

# ── Co-hub and group-specific hub gene lists ──────────────────────────────────
pd.DataFrame({'Co-hub':cohubs}).to_csv('bhub_true_cohubs.csv', index=False)
print('Saved: bhub_true_cohubs.csv')

# ── Download all files to local machine ──────────────────────────────────────
for fname in ['bhub_true_results.csv','bhub_true_edges.csv',
              'bhub_true_cohubs.csv','bhub_true_results.png',
              'bhub_diagnostics.png','bhub_edge_ppi.png']:
    try:
        files.download(fname)
    except Exception as e:
        print(f'  {fname}: {e}')